# Phase 3 · Rhetorical Role Prediction — Architecture-2 encoder (frozen) + IA3 + sentence BiLSTM + CRF

| | |
|---|---|
| **Codabench** | Macro-F1 **0.2789** · Macro-P 0.3675 · Macro-R 0.285 (rank 7 of 7 in Phase 3) |
| **Own dev / validation** | dev macro-F1: CL 0.3203, IT 0.3917, mean 0.3560 (best epoch 13) |
| **Backbone** | Architecture-2 encoder-decoder (150M) |

**The task, in simple words:** label **every sentence** of a judgment with its role — Fact, Issue, Argument, Statute, Precedent, Ratio, Ruling, ….

**How this notebook solves it:**
1. Tokenize every sentence; neighbouring sentences are read together so each one has context.
2. The Phase-1 model turns every sentence into one vector.
3. A BiLSTM reads the sentence vectors of the whole document in order, and a CRF picks the roles of all sentences together (roles follow patterns, e.g. facts come before the ratio).
4. Train on the train files; the dev files choose the best epoch; predict the test documents.

**What is special in this run:** **IA3** — the cheapest method: it only learns *scaling vectors* that turn the keys, values and feed-forward activations inside every block up or down (tens of thousands of weights instead of millions). They start at 1, i.e. no change.

*Exact setting:* IA3 (27,648 weights, 0.041% of the encoder).

**To run it:** Attach the tokenizer notebook's output and the Phase-1 checkpoint as Kaggle inputs, add the Hugging Face token as the Kaggle secret **`HF`**, pick a GPU (T4), then *Run All*. Paths are in the configuration cell; if a checkpoint is not found there, the notebook searches `/kaggle/input`. Test labels are never used.

Every step below starts with a short explanation: first in simple words, then how it works exactly.

### CELL 1 · Imports + reproducibility + device

**In simple words:** Load the Python libraries, fix the random seed (so a re-run gives the same result) and pick the GPU.

**How it works:** Seeds for `random`, NumPy and PyTorch; `DEVICE` is the GPU when one is attached.

In [1]:
# ============================================================
# CELL 1 — IMPORTS + REPRODUCIBILITY + DEVICE
# ============================================================

import os
import gc
import sys
import json
import math
import time
import random
import zipfile
import importlib.util

from pathlib import Path
from collections import Counter

import numpy as np

# less GPU memory fragmentation (must be set before importing torch)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import precision_recall_fscore_support


SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(SEED)

print("PyTorch:", torch.__version__)
print("Device :", DEVICE)

if DEVICE.type == "cuda":
    print("GPU    :", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
Device : cuda
GPU    : Tesla T4


### CELL 2 · Configuration

**In simple words:** Every setting of this run in one place: input paths, sizes, learning rates, epochs. To change the experiment, change only this cell.

**How it works:** The comment block at the top summarises the whole method. Mixed precision: float16 + GradScaler on T4 / P100, bfloat16 on A100. `DEBUG_MODE = True` (where present) runs a few-minute check on a small subset.

In [2]:
# ============================================================
# CELL 2 — CONFIGURATION
# Phase 3 (Rhetorical Roles): Architecture-2 ENCODER (frozen) + IA3
#                             + sentence vectors + BiLSTM + CRF
# Trained on TRAIN only; dev only picks the best epoch.
# ============================================================

RUN_NAME = "p3_arch2_ia3"


# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

TOKENIZER_DIR = Path(
    "/kaggle/input/notebooks/ritikguptatab/updated-bpe-tokenizer/"
    "legal_tokenizer_v3/tokenizer/final_tokenizer"
)

# Phase-1 Architecture-2 checkpoint (encoder-decoder; only the encoder is used).
# If wrong, CELL 9 searches /kaggle/input for it.
CHECKPOINT_PATH = Path(
    "/kaggle/input/models/ritikguptatab/proj-cap-p-1-archi-2/"
    "pytorch/default/1/best_encoder_decoder.pt"
)

# Phase-3 data: None = download from Hugging Face (Kaggle secret "HF"),
# or the folder that contains Train/ and Test/, e.g. Path("/kaggle/input/phase-3-rr/Phase_3_RR")
LOCAL_DATA_DIR = None
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"

OUTPUT_DIR = Path(f"/kaggle/working/{RUN_NAME}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# Sentences and windows
# (Architecture-2 encoder: learned positions, at most 1024 tokens)
# ------------------------------------------------------------

MAX_SENT_TOKENS = 256        # longer sentences are cut
WINDOW = 1024                # consecutive sentences are packed into windows of 1024 tokens
WINDOWS_PER_CHUNK = 8        # windows per training step (8 x 1024 tokens). If CUDA out-of-memory: 4

# ------------------------------------------------------------
# IA3 (Liu et al., 2022): learned rescaling vectors
#   keys   -> keys   * l_k        (k_proj output)
#   values -> values * l_v        (v_proj output)
#   FFN    -> fc2(hidden * l_ff)  (hidden = GELU(fc1(x)))
#   - Phase-1 weights stay FROZEN
#   - only l_k, l_v, l_ff are trained (they start at 1 -> no change)
# ------------------------------------------------------------

LR_ADAPTER = 3e-3            # IA3 has very few weights and needs a larger learning rate


# ------------------------------------------------------------
# Head: sentence vectors -> LayerNorm -> BiLSTM over the document -> CRF
# ------------------------------------------------------------

LSTM_HIDDEN = 256
HEAD_DROPOUT = 0.3
LR_HEAD = 2e-3

CLASS_BALANCING = True       # rare roles get a larger loss weight (macro-F1 counts every role equally)
CLASS_WEIGHT_MAX = 5.0       # cap, so a role with 3 sentences does not dominate
CE_WEIGHT = 0.5              # loss = CRF + CE_WEIGHT * class-weighted cross-entropy

# A role with fewer than MIN_LABEL_COUNT train sentences in a domain is never
# predicted in that domain (e.g. Dissent in IT: 0 sentences). Predicting a role
# that is not in the gold adds a 0 to the macro-F1 average.
MIN_LABEL_COUNT = 10


# ------------------------------------------------------------
# Training (TRAIN data only)
# ------------------------------------------------------------

EPOCHS = 15
PATIENCE = 5                 # stop if the dev score does not improve for 5 epochs
HEAD_ONLY_EPOCHS = 1         # epoch 1 trains only the head (the Phase-1 encoder is used as it is)
WARMUP_RATIO = 0.06
GRAD_CLIP = 1.0
TRAIN_EVAL_DOCS = 10         # a few train documents are scored every epoch (to see under-fitting)


# ------------------------------------------------------------
# Debug mode (quick run on a few documents)
# ------------------------------------------------------------

DEBUG_MODE = False
DEBUG_DOCS = 6               # per file

if DEBUG_MODE:
    EPOCHS = 2


# ------------------------------------------------------------
# Mixed precision (T4 / P100 -> float16 + GradScaler, A100 -> bfloat16)
# ------------------------------------------------------------

USE_AMP = DEVICE.type == "cuda"

if USE_AMP and torch.cuda.get_device_capability()[0] >= 8:
    AMP_DTYPE = torch.bfloat16
else:
    AMP_DTYPE = torch.float16

USE_SCALER = USE_AMP and AMP_DTYPE == torch.float16


def autocast():
    return torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP)


assert MAX_SENT_TOKENS <= WINDOW

print("Run       :", RUN_NAME)
print("Output    :", OUTPUT_DIR)
print("AMP dtype :", AMP_DTYPE if USE_AMP else "disabled")

Run       : p3_arch2_ia3
Output    : /kaggle/working/p3_arch2_ia3
AMP dtype : torch.float16


### CELL 3 · Load the Phase-1 tokenizer

**In simple words:** Load our own Phase-1 BPE tokenizer — the same one the Phase-1 model was trained with. Any other tokenizer would give the model token ids it has never seen.

**How it works:** `tokenizer.py` from the tokenizer notebook. Long documents: only the needed part is tokenized, and every pre-token piece is BPE-encoded once and cached, which makes tokenizing thousands of judgments fast.

In [3]:
# ============================================================
# CELL 3 — LOAD THE PHASE-1 TOKENIZER
# ============================================================

spec = importlib.util.spec_from_file_location("legal_tokenizer_module", TOKENIZER_DIR / "tokenizer.py")
tok_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(tok_module)

tokenizer, tokenizer_config = tok_module.load_tokenizer(TOKENIZER_DIR)

PAD_ID = tokenizer.pad_id
UNK_ID = tokenizer.unk_id
VOCAB_SIZE = tokenizer.get_vocab_size()

print("Vocabulary size:", VOCAB_SIZE, "| PAD / UNK:", PAD_ID, UNK_ID)


token_cache = {}      # sentence -> token ids (legal documents repeat many sentences)


def encode_sentence(sentence):
    """
    Token ids of one sentence (NOT cut yet; CELL 6 cuts to MAX_SENT_TOKENS).
    A space is added in front: inside a document every sentence follows a
    space, and the tokenizer keeps the space in front of the first word.
    """

    if sentence not in token_cache:

        ids = tokenizer.encode(" " + sentence.strip())

        token_cache[sentence] = ids if ids else [UNK_ID]   # empty sentence -> 1 token, so it still gets a vector

    return token_cache[sentence]


demo = "The petitioner filed the present appeal."
print("\nDemo:", [tokenizer.id_to_token[i] for i in encode_sentence(demo)])

Vocabulary size: 32004 | PAD / UNK: 0 1

Demo: [' ', 'The', ' ', 'petitioner', ' ', 'filed', ' ', 'the', ' ', 'present', ' ', 'appeal', '.']


### CELL 4 · Phase-3 data files (download needs the Kaggle secret "HF")

**In simple words:** Download the official data of this phase from Hugging Face.

**How it works:** The dataset is gated: the token comes from the Kaggle secret `HF` (never written in the notebook). Files are cached, so a re-run does not download again.

In [4]:
# ============================================================
# CELL 4 — PHASE-3 DATA FILES  (download needs the Kaggle secret "HF")
# ============================================================

PHASE3_FILES = {
    "CL_train": "Phase_3_RR/Train/CL_train.json",
    "CL_dev":   "Phase_3_RR/Train/CL_dev.json",
    "IT_train": "Phase_3_RR/Train/IT_train.json",
    "IT_dev":   "Phase_3_RR/Train/IT_dev.json",
    "labels":   "Phase_3_RR/Train/label_vocab.json",
    "CL_test":  "Phase_3_RR/Test/CL_test.json",
    "IT_test":  "Phase_3_RR/Test/IT_test.json",
}

DATA_FILES = {}

if LOCAL_DATA_DIR is not None:

    for name, filename in PHASE3_FILES.items():
        DATA_FILES[name] = Path(LOCAL_DATA_DIR) / filename.replace("Phase_3_RR/", "")

else:

    from huggingface_hub import hf_hub_download
    from kaggle_secrets import UserSecretsClient

    HF_TOKEN = UserSecretsClient().get_secret("HF")

    for name, filename in PHASE3_FILES.items():

        DATA_FILES[name] = hf_hub_download(
            repo_id=HF_DATASET_REPO,
            filename=filename,
            repo_type="dataset",
            token=HF_TOKEN,
        )

for name, path in DATA_FILES.items():
    print(f"{name:9s} -> {path}")

CL_train  -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_3_RR/Train/CL_train.json
CL_dev    -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_3_RR/Train/CL_dev.json
IT_train  -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_3_RR/Train/IT_train.json
IT_dev    -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_3_RR/Train/IT_dev.json
labels    -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_3_RR/Train/label_vocab.json
CL_test   -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_3_RR/Test/CL_test.json
I

### CELL 5 · Load documents + labels + statistics

**In simple words:** Read the documents of both domains (Corporate Law, IT): every document is a list of sentences, each with one role.

**How it works:** Also the label list, and which roles occur in each domain (others are never predicted for that domain).

In [5]:
# ============================================================
# CELL 5 — LOAD DOCUMENTS + LABELS + STATISTICS
#
# Every document: {"id", "domain", "sentences": [...], "labels": [...] or None}
# ============================================================

def read_json(path):
    """A JSON file, or a JSON-lines file (one document per line)."""

    with open(path, "r", encoding="utf-8") as f:
        content = f.read()

    try:
        return json.loads(content)
    except json.JSONDecodeError:
        return [json.loads(line) for line in content.splitlines() if line.strip()]


def load_label_vocab(path):

    raw = read_json(path)

    if isinstance(raw, dict) and isinstance(raw.get("labels"), list):
        raw = raw["labels"]

    if isinstance(raw, list):
        return [str(label) for label in raw]

    # dict: {label: id} or {id: label}
    if all(isinstance(v, int) for v in raw.values()):
        return [label for label, _ in sorted(raw.items(), key=lambda item: item[1])]

    return [str(raw[key]) for key in sorted(raw, key=int)]


def load_documents(path, domain):

    raw = read_json(path)

    # accepted layouts: [doc, ...]  |  {"data": [doc, ...]}  |  {doc_id: doc}
    if isinstance(raw, dict):

        list_key = next((k for k in ("data", "documents", "docs") if isinstance(raw.get(k), list)), None)

        if list_key is not None:
            raw = raw[list_key]
        else:
            raw = [
                dict(value, id=key) if isinstance(value, dict) else {"id": key, "text": value}
                for key, value in raw.items()
            ]

    docs = []

    for i, item in enumerate(raw):

        sentences = item.get("text", item.get("sentences"))
        labels = item.get("labels", item.get("label"))

        doc = {
            "id": str(item.get("id", f"{domain}_{i}")),
            "domain": domain,
            "sentences": [str(s) for s in sentences],
            "labels": None if labels is None else [str(label) for label in labels],
        }

        if doc["labels"] is not None:
            assert len(doc["labels"]) == len(doc["sentences"]), f"{doc['id']}: labels and sentences differ in length"

        docs.append(doc)

    if DEBUG_MODE:
        docs = docs[:DEBUG_DOCS]

    return docs


LABELS = load_label_vocab(DATA_FILES["labels"])
LABEL2ID = {label: i for i, label in enumerate(LABELS)}
NUM_LABELS = len(LABELS)

print(f"{NUM_LABELS} labels:", LABELS)

train_docs = load_documents(DATA_FILES["CL_train"], "CL") + load_documents(DATA_FILES["IT_train"], "IT")
dev_docs = load_documents(DATA_FILES["CL_dev"], "CL") + load_documents(DATA_FILES["IT_dev"], "IT")
test_docs = load_documents(DATA_FILES["CL_test"], "CL") + load_documents(DATA_FILES["IT_test"], "IT")


# ------------------------------------------------------------
# Label ids. A label that is not in label_vocab.json is reported
# and mapped to "None".
# ------------------------------------------------------------

unknown = Counter()

for doc in train_docs + dev_docs:

    for k, label in enumerate(doc["labels"]):
        if label not in LABEL2ID:
            unknown[label] += 1
            doc["labels"][k] = "None"

    doc["label_ids"] = [LABEL2ID[label] for label in doc["labels"]]

if unknown:
    print("\nWARNING: labels not in label_vocab.json (mapped to 'None'):", dict(unknown))


# ------------------------------------------------------------
# Statistics
# ------------------------------------------------------------

print(f"\n{'split':6s} {'domain':6s} {'docs':>5s} {'sentences':>10s} {'per doc (mean / max)':>22s}")

for split, docs in [("train", train_docs), ("dev", dev_docs), ("test", test_docs)]:
    for domain in ["CL", "IT"]:

        lengths = [len(d["sentences"]) for d in docs if d["domain"] == domain]

        if lengths:
            print(f"{split:6s} {domain:6s} {len(lengths):5d} {sum(lengths):10d} "
                  f"{np.mean(lengths):14.1f} / {max(lengths):5d}")

print(f"\nLabel distribution (train | dev):")
print(f"{'label':24s} {'CL train':>9s} {'IT train':>9s} {'CL dev':>7s} {'IT dev':>7s}")

counts = {
    (split, domain): Counter(label for d in docs if d["domain"] == domain for label in d["labels"])
    for split, docs in [("train", train_docs), ("dev", dev_docs)]
    for domain in ["CL", "IT"]
}

for label in LABELS:
    print(f"{label:24s} {counts['train', 'CL'][label]:9d} {counts['train', 'IT'][label]:9d} "
          f"{counts['dev', 'CL'][label]:7d} {counts['dev', 'IT'][label]:7d}")


# ------------------------------------------------------------
# Class weights for the loss: (N / (C * n_label)) ** 0.5, at most CLASS_WEIGHT_MAX.
# The square root keeps the weights moderate.
# ------------------------------------------------------------

train_counts = Counter(label for d in train_docs for label in d["labels"])
n_train = sum(train_counts.values())

if CLASS_BALANCING:
    CLASS_WEIGHTS = torch.tensor(
        [min(CLASS_WEIGHT_MAX, (n_train / (NUM_LABELS * max(train_counts[label], 1))) ** 0.5) for label in LABELS]
    )
else:
    CLASS_WEIGHTS = torch.ones(NUM_LABELS)

print("\nClass weights:", {label: round(w, 2) for label, w in zip(LABELS, CLASS_WEIGHTS.tolist())})


# ------------------------------------------------------------
# Roles allowed per domain (used at prediction time, CELL 10)
# ------------------------------------------------------------

LABEL_ALLOWED = {
    domain: np.array([counts["train", domain][label] >= MIN_LABEL_COUNT for label in LABELS])
    for domain in ["CL", "IT"]
}

for domain in ["CL", "IT"]:
    never = [label for label, ok in zip(LABELS, LABEL_ALLOWED[domain]) if not ok]
    print(f"Never predicted in {domain} (< {MIN_LABEL_COUNT} train sentences):", never)
print("\nSample:", train_docs[0]["id"], "|", train_docs[0]["labels"][0], "|", train_docs[0]["sentences"][0][:120])


split  domain  docs  sentences   per doc (mean / max)
train  CL        40      10593          264.8 /  1816
train  IT        40       6252          156.3 /   442
dev    CL         5       1396          279.2 /   848
dev    IT         5        746          149.2 /   382
test   CL         5       1339          267.8 /   942
test   IT         5        858          171.6 /   420

Label distribution (train | dev):
label                     CL train  IT train  CL dev  IT dev
Fact                          2979      1119     249     195
Issue                           17        90       3      12
ArgumentPetitioner             976       637     344      56
ArgumentRespondent             996       352     236      36
Statute                        367       732      58      62
Dissent                        284         0       0       0
PrecedentReliedUpon           1305       640     142     112
PrecedentNotReliedUpon         209       360       0      50
PrecedentOverruled               0   

### CELL 6 · Tokenize + windows + chunks

**In simple words:** Tokenize every sentence and pack neighbouring sentences into windows, so each sentence is read together with the sentences before it.

**How it works:** `CONTEXT = "document"`: up to `WINDOW` tokens per window (Architecture-1 is causal, so a sentence sees what came before). A *chunk* = the windows of one training step, consecutive sentences of one document.

In [6]:
# ============================================================
# CELL 6 — TOKENIZE + WINDOWS + CHUNKS
#
# window = token ids that go through the encoder together,
#          {"ids": [...], "sentences": [(sentence_index, start, end), ...]}
#   consecutive sentences are packed into one window (up to WINDOW
#   tokens). The encoder is bidirectional, so every sentence sees the
#   sentences before AND after it in its window.
#
# chunk = the windows of ONE training step (consecutive sentences
#         of one document).
# ============================================================

def pack_windows(doc):

    windows = []

    for i, sentence in enumerate(doc["sentences"]):

        ids = encode_sentence(sentence)[:MAX_SENT_TOKENS]

        if not windows or len(windows[-1]["ids"]) + len(ids) > WINDOW:
            windows.append({"ids": [], "sentences": []})

        window = windows[-1]

        window["sentences"].append((i, len(window["ids"]), len(window["ids"]) + len(ids)))
        window["ids"] = window["ids"] + ids

    return windows


def make_chunks(docs):

    chunks = []

    for doc in docs:
        for k in range(0, len(doc["windows"]), WINDOWS_PER_CHUNK):
            chunks.append({"doc": doc, "windows": doc["windows"][k:k + WINDOWS_PER_CHUNK]})

    return chunks


t0 = time.time()

for doc in train_docs + dev_docs + test_docs:

    doc["windows"] = pack_windows(doc)

    # every sentence must be in exactly one window, in order
    order = [i for window in doc["windows"] for i, _, _ in window["sentences"]]
    assert order == list(range(len(doc["sentences"]))), doc["id"]

print(f"Tokenized in {time.time() - t0:.1f}s")

lengths = np.array([len(encode_sentence(s)) for d in train_docs for s in d["sentences"]])

print(f"Sentences longer than {MAX_SENT_TOKENS} tokens (cut): {(lengths > MAX_SENT_TOKENS).mean():.2%}")
print(f"Train windows: {sum(len(d['windows']) for d in train_docs)} | "
      f"train chunks (steps per epoch): {len(make_chunks(train_docs))}")

Tokenized in 6.6s
Sentences longer than 256 tokens (cut): 1.92%
Train windows: 1435 | train chunks (steps per epoch): 212


### CELL 7 · Scores + Viterbi

**In simple words:** The scoring rule (macro-F1: every role counts equally, however rare) and Viterbi decoding for the CRF.

In [7]:
# ============================================================
# CELL 7 — SCORES + VITERBI
# ============================================================

def macro_scores(gold, pred):
    """Macro precision / recall / F1 over the labels that occur in gold or prediction."""

    labels = sorted(set(gold) | set(pred))

    p, r, f, _ = precision_recall_fscore_support(gold, pred, labels=labels, average="macro", zero_division=0)

    return {"precision": float(p), "recall": float(r), "f1": float(f)}


def flat_labels(docs, predictions, domain=None):
    """Gold and predicted labels of all sentences, in order (one domain or all)."""

    gold, pred = [], []

    for doc in docs:
        if domain is None or doc["domain"] == domain:
            gold.extend(doc["labels"])
            pred.extend(predictions[doc["id"]])

    return gold, pred


def domain_scores(docs, predictions):
    """
    Macro scores for CL, IT and all sentences together.
    "score" = mean of the CL and IT macro-F1 (both domains count equally);
    it is used to pick the best epoch.
    """

    scores = {}

    for domain in ["CL", "IT"]:

        gold, pred = flat_labels(docs, predictions, domain)

        if gold:
            scores[domain] = macro_scores(gold, pred)

    scores["all"] = macro_scores(*flat_labels(docs, predictions))
    scores["score"] = float(np.mean([scores[d]["f1"] for d in ["CL", "IT"] if d in scores]))

    return scores


def viterbi(emissions, transitions, start_scores, end_scores):
    """emissions [n_sentences, NUM_LABELS] -> best label id per sentence (numpy)."""

    n = len(emissions)

    score = start_scores + emissions[0]
    backpointer = np.zeros((n, NUM_LABELS), dtype=np.int64)

    for i in range(1, n):
        candidates = score[:, None] + transitions          # [previous, current]
        backpointer[i] = candidates.argmax(axis=0)
        score = candidates.max(axis=0) + emissions[i]

    path = [int((score + end_scores).argmax())]

    for i in range(n - 1, 0, -1):
        path.append(int(backpointer[i, path[-1]]))

    return np.array(path[::-1], dtype=np.int64)


_g = ["Fact", "Fact", "Issue", "Statute"]
_p = ["Fact", "Issue", "Issue", "Statute"]
print("macro_scores check:", macro_scores(_g, _p))

macro_scores check: {'precision': 0.8333333333333334, 'recall': 0.8333333333333334, 'f1': 0.7777777777777777}


### CELL 8 · Model

**In simple words:** The Phase-1 model turns every sentence into one vector; a BiLSTM reads the sentence vectors of the whole document in order, and a CRF chooses the roles of all sentences together.

**How it works:** sentence vector = mean of its token vectors + relative position in the document + domain flag. The CRF learns which role follows which. Adaptation: IA3 (27,648 weights, 0.041% of the encoder).

In [8]:
# ============================================================
# CELL 8 — MODEL
#
#   sentences -> Architecture-2 ENCODER (Phase-1 weights FROZEN, + IA3)
#             -> mean of the token vectors of each sentence = sentence vector
#             -> LayerNorm, + relative position in the document + domain flag
#             -> BiLSTM over the sentences of the document -> CRF
#
# Layer names match the Architecture-2 notebook, so the weights load directly.
# ============================================================

# ============================================================
# Architecture-2 ENCODER
# ============================================================

class MultiHeadAttention(nn.Module):

    def __init__(self, d_model, n_heads, dropout):
        super().__init__()

        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.dropout = dropout

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

    def split_heads(self, x):
        B, T, D = x.shape
        return x.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

    def forward(self, x, attn_mask):

        q, k, v = self.split_heads(self.q_proj(x)), self.split_heads(self.k_proj(x)), self.split_heads(self.v_proj(x))

        out = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask,
                                             dropout_p=self.dropout if self.training else 0.0)
        B, H, T, d = out.shape

        return self.out_proj(out.transpose(1, 2).reshape(B, T, H * d))


class PositionalEmbedding(nn.Module):

    def __init__(self, max_len, d_model, dropout):
        super().__init__()
        self.embedding = nn.Embedding(max_len, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        positions = torch.arange(x.size(1), device=x.device)
        return self.dropout(x + self.embedding(positions).unsqueeze(0))


class FeedForward(nn.Module):

    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.fc2(F.gelu(self.fc1(x))))


class EncoderBlock(nn.Module):

    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.self_attention = MultiHeadAttention(d_model, n_heads, dropout)
        self.norm2 = nn.LayerNorm(d_model)
        self.feed_forward = FeedForward(d_model, d_ff, dropout)

    def forward(self, x, attn_mask):
        x = x + self.self_attention(self.norm1(x), attn_mask)
        return x + self.feed_forward(self.norm2(x))


class Arch2Encoder(nn.Module):
    """Architecture-2 encoder: bidirectional, learned positions."""

    def __init__(self, cfg):
        super().__init__()

        self.token_embedding = nn.Embedding(cfg["vocab_size"], cfg["d_model"])
        self.position_embedding = PositionalEmbedding(cfg["src_len"], cfg["d_model"], cfg["dropout"])
        self.layers = nn.ModuleList([
            EncoderBlock(cfg["d_model"], cfg["n_heads"], cfg["ffn_dim"], cfg["dropout"])
            for _ in range(cfg["encoder_layers"])
        ])
        self.final_norm = nn.LayerNorm(cfg["d_model"])

    def forward(self, input_ids, mask):

        attn_mask = mask[:, None, None, :]                    # padding is never attended to
        x = self.position_embedding(self.token_embedding(input_ids))

        for layer in self.layers:
            x = layer(x, attn_mask)

        return self.final_norm(x)


# ============================================================
# IA3 (learned rescaling vectors)
#   IA3Output: layer(x) * l     -> used for keys (k_proj) and values (v_proj)
#   IA3Input : layer(x * l)     -> used for the FFN hidden, the input of fc2
# The vectors start at 1, so the model starts unchanged.
# ============================================================

class IA3Output(nn.Module):

    def __init__(self, base):
        super().__init__()

        self.base = base                                      # frozen Phase-1 layer
        self.ia3_scale = nn.Parameter(torch.ones(base.out_features))

    def forward(self, x):

        h = self.base(x)

        return h * self.ia3_scale.to(h.dtype)


class IA3Input(nn.Module):

    def __init__(self, base):
        super().__init__()

        self.base = base                                      # frozen Phase-1 layer
        self.ia3_scale = nn.Parameter(torch.ones(base.in_features))

    def forward(self, x):

        return self.base(x * self.ia3_scale.to(x.dtype))


def add_adapters(encoder):
    """Freeze every Phase-1 weight, then add the IA3 vectors to every encoder block."""

    for p in encoder.parameters():
        p.requires_grad = False

    for layer in encoder.layers:

        attention = layer.self_attention
        attention.k_proj = IA3Output(attention.k_proj)        # l_k
        attention.v_proj = IA3Output(attention.v_proj)        # l_v

        layer.feed_forward.fc2 = IA3Input(layer.feed_forward.fc2)   # l_ff on GELU(fc1(x))


# ============================================================
# CRF + sentence-role model
# ============================================================

class CRF(nn.Module):
    """Linear-chain CRF over the sentences of a document (learns which role follows which)."""

    def __init__(self, num_labels):
        super().__init__()

        self.transitions = nn.Parameter(torch.zeros(num_labels, num_labels))      # [previous, current]
        self.start = nn.Parameter(torch.zeros(num_labels))
        self.end = nn.Parameter(torch.zeros(num_labels))

    def loss(self, emissions, labels):
        """Negative log-likelihood per sentence. emissions [S, C] (float32), labels [S]."""

        S = emissions.size(0)

        gold = (
            self.start[labels[0]]
            + emissions[torch.arange(S, device=emissions.device), labels].sum()
            + self.transitions[labels[:-1], labels[1:]].sum()
            + self.end[labels[-1]]
        )

        alpha = self.start + emissions[0]

        for i in range(1, S):
            alpha = torch.logsumexp(alpha.unsqueeze(1) + self.transitions, dim=0) + emissions[i]

        log_z = torch.logsumexp(alpha + self.end, dim=0)

        return (log_z - gold) / S


class SentenceRoleModel(nn.Module):
    """Architecture-2 encoder (+ adapters) + head."""

    def __init__(self, backbone, d_model):
        super().__init__()

        self.backbone = backbone
        self.dropout = nn.Dropout(HEAD_DROPOUT)

        # LayerNorm: sentence vectors of any size -> mean 0, std 1 (keeps the BiLSTM from saturating)
        self.vector_norm = nn.LayerNorm(d_model)

        self.lstm = nn.LSTM(d_model + 2, LSTM_HIDDEN, batch_first=True, bidirectional=True)
        self.classifier = nn.Linear(2 * LSTM_HIDDEN, NUM_LABELS)
        self.crf = CRF(NUM_LABELS)

        self.register_buffer("class_weights", CLASS_WEIGHTS.clone())

    def head_parameters(self):
        return [p for name, p in self.named_parameters() if not name.startswith("backbone.")]

    def sentence_vectors(self, windows):
        """windows -> (vectors [S, D] float32, sentence_index [S]). Sentence vector = mean of its token vectors."""

        max_len = max(len(w["ids"]) for w in windows)
        ids = torch.full((len(windows), max_len), PAD_ID, dtype=torch.long)
        mask = torch.zeros((len(windows), max_len), dtype=torch.bool)

        for row, window in enumerate(windows):
            ids[row, :len(window["ids"])] = torch.tensor(window["ids"])
            mask[row, :len(window["ids"])] = True

        with autocast():
            hidden = self.backbone(ids.to(DEVICE), mask.to(DEVICE))           # [W, T, D]

        vectors = []
        sentence_index = []

        for row, window in enumerate(windows):
            for i, start, end in window["sentences"]:
                vectors.append(hidden[row, start:end].float().mean(dim=0))
                sentence_index.append(i)

        return torch.stack(vectors), sentence_index

    def head(self, vectors, doc, sentence_index):
        """Sentence vectors [S, D] -> label scores [S, NUM_LABELS] (float32)."""

        n = len(doc["sentences"])

        position = torch.tensor([[i / max(n - 1, 1)] for i in sentence_index], device=DEVICE)
        domain = torch.full_like(position, 1.0 if doc["domain"] == "IT" else 0.0)

        x = self.dropout(torch.cat([self.vector_norm(vectors), position, domain], dim=-1))

        out, _ = self.lstm(x.unsqueeze(0))              # one sequence = the sentences in document order

        return self.classifier(self.dropout(out.squeeze(0)))

    def chunk_emissions(self, chunk):
        """Label scores of the sentences of one training chunk."""

        vectors, sentence_index = self.sentence_vectors(chunk["windows"])

        return self.head(vectors, chunk["doc"], sentence_index), sentence_index


# ------------------------------------------------------------
# Check: IA3 really rescales
# ------------------------------------------------------------

_x = torch.randn(3, 8)

_out = IA3Output(nn.Linear(8, 6))
nn.init.normal_(_out.ia3_scale)
assert torch.allclose(_out(_x), _out.base(_x) * _out.ia3_scale, atol=1e-6)

_inp = IA3Input(nn.Linear(8, 6))
nn.init.normal_(_inp.ia3_scale)
assert torch.allclose(_inp(_x), _inp.base(_x * _inp.ia3_scale), atol=1e-6)

print("✅ IA3: k * l_k, v * l_v, fc2(hidden * l_ff)")

✅ IA3: k * l_k, v * l_v, fc2(hidden * l_ff)


### CELL 9 · Build the model from the Phase-1 Architecture-2 checkpoint

**In simple words:** Create the network, load the trained Phase-1 weights into it, and add the adaptation (the part that will learn).

**How it works:** The layer names match the Phase-1 notebook, so the checkpoint loads directly; if it is not at the configured path, the notebook searches `/kaggle/input`. The printed counts show how many weights train. Here: IA3 (27,648 weights, 0.041% of the encoder).

In [9]:
# ============================================================
# CELL 9 — BUILD THE MODEL FROM THE PHASE-1 ARCHITECTURE-2 CHECKPOINT
# ============================================================

if not CHECKPOINT_PATH.exists():

    found = []
    for path in sorted(Path("/kaggle/input").rglob("best_encoder_decoder.pt")):
        try:
            if "encoder_layers" in torch.load(path, map_location="cpu", weights_only=False).get("config", {}):
                found.append(path)
        except Exception:
            pass

    assert found, f"{CHECKPOINT_PATH} not found and no Architecture-2 checkpoint under /kaggle/input"

    CHECKPOINT_PATH = found[0]
    print("Checkpoint found at:", CHECKPOINT_PATH)


phase1 = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
ENCODER_CONFIG = dict(phase1["config"])

# encoder weights only (the decoder of Architecture-2 is not needed)
ENCODER_WEIGHTS = {}

for key, value in phase1["model_state_dict"].items():
    key = key.replace("_orig_mod.", "")
    if key.startswith("encoder."):
        ENCODER_WEIGHTS[key[len("encoder."):]] = value

del phase1

print("Encoder config:", {k: ENCODER_CONFIG[k] for k in ("d_model", "encoder_layers", "n_heads", "ffn_dim", "src_len", "dropout")})

assert ENCODER_CONFIG["vocab_size"] == VOCAB_SIZE, "checkpoint was trained with a different tokenizer"
assert WINDOW <= ENCODER_CONFIG["src_len"], f"WINDOW must be <= {ENCODER_CONFIG['src_len']} (learned positions)"


def build_model():
    """Frozen Phase-1 encoder + IA3 + new head."""

    encoder = Arch2Encoder(ENCODER_CONFIG)
    encoder.load_state_dict(ENCODER_WEIGHTS, strict=True)          # every name and shape must match

    # the adapters start as "no change": the output must stay the same
    test_ids = torch.randint(4, VOCAB_SIZE, (1, 24))
    test_mask = torch.ones(1, 24, dtype=torch.bool)

    encoder.eval()
    with torch.no_grad():
        before = encoder(test_ids, test_mask)
        add_adapters(encoder)
        after = encoder(test_ids, test_mask)
    encoder.train()

    assert torch.allclose(before, after, atol=1e-4), "IA3 changed the model at the start"

    return SentenceRoleModel(encoder, ENCODER_CONFIG["d_model"]).to(DEVICE)


# free GPU memory if this cell is re-run
for name in ["model", "optimizer", "scaler"]:
    globals().pop(name, None)

sys.last_traceback = sys.last_value = sys.last_type = None
gc.collect()

if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

model = build_model()

n_total = sum(p.numel() for p in model.parameters())
n_encoder = sum(p.numel() for p in model.backbone.parameters())
n_adapter = sum(p.numel() for p in model.backbone.parameters() if p.requires_grad)
n_head = sum(p.numel() for p in model.head_parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

assert n_trainable == n_adapter + n_head

print(f"\nPhase-1 encoder (frozen) : {n_encoder - n_adapter:,}")
print(f"IA3 (trainable)          : {n_adapter:,}  ({n_adapter / (n_encoder - n_adapter):.3%} of the encoder)")
print(f"Head (trainable)         : {n_head:,}")
print(f"Trainable total          : {n_trainable:,}  ({n_trainable / n_total:.2%} of {n_total:,})")


# ------------------------------------------------------------
# Size of the sentence vectors BEFORE the LayerNorm of the head
# ------------------------------------------------------------

model.eval()
with torch.no_grad():
    _vectors, _ = model.sentence_vectors(make_chunks(train_docs[:1])[0]["windows"])
model.train()

_cos = F.cosine_similarity(_vectors[:, None], _vectors[None], dim=-1).mean()

print(f"\nSentence vectors: mean |value| {_vectors.abs().mean():.2f} | max |value| {_vectors.abs().max():.1f} | "
      f"mean cosine between sentences {_cos:.3f}")

Checkpoint found at: /kaggle/input/notebooks/ritikguptatab/proj-cap-p-1-archi-2/architecture_2_encoder_decoder/best_encoder_decoder.pt
Encoder config: {'d_model': 768, 'encoder_layers': 6, 'n_heads': 12, 'ffn_dim': 3072, 'src_len': 1024, 'dropout': 0.1}

Phase-1 encoder (frozen) : 67,894,272
IA3 (trainable)          : 27,648  (0.041% of the encoder)
Head (trainable)         : 2,113,744
Trainable total          : 2,141,392  (3.06% of 70,035,664)

Sentence vectors: mean |value| 0.05 | max |value| 2.2 | mean cosine between sentences 0.898


### CELL 10 · Prediction + evaluation

**In simple words:** Predict the roles of whole documents and score them.

**How it works:** The head sees all sentences of a document at once and the CRF's Viterbi picks the role sequence; roles never seen in the document's domain are not allowed; every sentence gets exactly one role.

In [10]:
# ============================================================
# CELL 10 — PREDICTION + EVALUATION 
#
# A whole document goes through the model (windows in batches),
# then the head sees all its sentences at once and the CRF Viterbi
# picks the best role sequence. Roles not allowed in the document's
# domain (CELL 5) are never picked. One role per sentence.
# ============================================================

EVAL_WINDOWS = 8          # windows per forward pass


@torch.no_grad()
def document_emissions(model, doc):
    """Label scores of every sentence of one document: numpy [n_sentences, NUM_LABELS].
    The model must be in eval mode."""

    vectors, sentence_index = [], []

    for k in range(0, len(doc["windows"]), EVAL_WINDOWS):

        v, idx = model.sentence_vectors(doc["windows"][k:k + EVAL_WINDOWS])

        vectors.append(v)
        sentence_index.extend(idx)

    return model.head(torch.cat(vectors), doc, sentence_index).float().cpu().numpy()


@torch.no_grad()
def predict_labels(model, docs):
    """{doc_id: [label, label, ...]}  one label per sentence, in order."""

    model.eval()

    transitions = model.crf.transitions.float().cpu().numpy()
    start = model.crf.start.float().cpu().numpy()
    end = model.crf.end.float().cpu().numpy()

    predictions = {}

    for doc in docs:

        if not doc["sentences"]:
            predictions[doc["id"]] = []
            continue

        emissions = document_emissions(model, doc)
        emissions[:, ~LABEL_ALLOWED[doc["domain"]]] = -1e4

        predictions[doc["id"]] = [LABELS[i] for i in viterbi(emissions, transitions, start, end)]

    model.train()

    return predictions


def evaluate(model, docs):

    predictions = predict_labels(model, docs)

    return domain_scores(docs, predictions), predictions

### CELL 11 · Loss + optimizer + training function

**In simple words:** Loss, optimizer and training loop.

**How it works:** loss = CRF + 0.5 · class-weighted cross-entropy (rare roles weigh more, because macro-F1 counts every role equally).

In [11]:
# ============================================================
# CELL 11 — LOSS + OPTIMIZER + TRAINING FUNCTION
#   loss = CRF + CE_WEIGHT * class-weighted cross-entropy
#   group 0 = IA3 weights, group 1 = head
# ============================================================

def chunk_loss(model, chunk):

    doc = chunk["doc"]

    emissions, sentence_index = model.chunk_emissions(chunk)
    labels = torch.tensor([doc["label_ids"][i] for i in sentence_index], device=DEVICE)

    return model.crf.loss(emissions, labels) + CE_WEIGHT * F.cross_entropy(emissions, labels, weight=model.class_weights)


def make_optimizer(model, total_steps):

    optimizer = torch.optim.AdamW(
        [
            {"params": [p for p in model.backbone.parameters() if p.requires_grad], "lr": LR_ADAPTER, "weight_decay": 0.0},
            {"params": model.head_parameters(), "lr": LR_HEAD, "weight_decay": 0.0},
        ],
        betas=(0.9, 0.999),
    )

    base_lrs = [group["lr"] for group in optimizer.param_groups]
    warmup_steps = max(1, int(WARMUP_RATIO * total_steps))

    def set_lr(step):
        """Linear warmup, then linear decay to 0."""

        if step < warmup_steps:
            factor = (step + 1) / warmup_steps
        else:
            factor = max(0.0, (total_steps - step) / max(1, total_steps - warmup_steps))

        for group, base_lr in zip(optimizer.param_groups, base_lrs):
            group["lr"] = base_lr * factor

    return optimizer, set_lr


def trainable_state(model):
    """Only the trained weights (IA3 + head): small, quick to copy."""

    return {name: p.detach().cpu().clone() for name, p in model.named_parameters() if p.requires_grad}


def train_model(model, epochs):
    """
    Trains on TRAIN documents only. After every epoch the model is scored on dev;
    the best epoch is kept, and training stops after PATIENCE epochs without improvement.
    Returns (history, best_epoch).
    """

    chunks = make_chunks(train_docs)

    optimizer, set_lr = make_optimizer(model, epochs * len(chunks))
    scaler = torch.amp.GradScaler("cuda", enabled=USE_SCALER)

    train_check_docs = random.Random(SEED).sample(train_docs, min(TRAIN_EVAL_DOCS, len(train_docs)))

    history = []
    best_score, best_epoch, best_state = -1.0, 0, None
    step = 0
    t_start = time.time()

    model.train()

    for epoch in range(1, epochs + 1):

        random.shuffle(chunks)
        total_loss = 0.0
        skipped = 0                   # steps skipped by the float16 GradScaler (overflow)
        head_only = epoch <= HEAD_ONLY_EPOCHS

        for chunk in chunks:

            set_lr(step)

            if head_only:
                optimizer.param_groups[0]["lr"] = 0.0          # group 0 = adapter: not trained yet

            loss = chunk_loss(model, chunk)

            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)

            # clip adapter and head separately
            torch.nn.utils.clip_grad_norm_(optimizer.param_groups[0]["params"], GRAD_CLIP)
            torch.nn.utils.clip_grad_norm_(optimizer.param_groups[1]["params"], GRAD_CLIP)

            scale_before = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()
            skipped += scaler.get_scale() < scale_before

            step += 1
            total_loss += loss.item()

        train_scores, _ = evaluate(model, train_check_docs)
        dev_scores, _ = evaluate(model, dev_docs)

        log = {"epoch": epoch, "loss": total_loss / len(chunks), "train_f1": train_scores["score"],
               "dev": dev_scores, "head_only": head_only, "skipped_steps": int(skipped)}

        line = (f"Epoch {epoch:2d}/{epochs}{' (head only)' if head_only else ''} | loss {log['loss']:.4f} | "
                f"train F1 {log['train_f1']:.4f} | dev F1 CL {dev_scores.get('CL', {}).get('f1', 0):.4f} "
                f"IT {dev_scores.get('IT', {}).get('f1', 0):.4f} mean {dev_scores['score']:.4f}")

        stop = False

        if dev_scores["score"] > best_score:
            best_score, best_epoch, best_state = dev_scores["score"], epoch, trainable_state(model)
            line += "  <- best"
        elif epoch - best_epoch >= PATIENCE:
            stop = True

        if skipped:
            line += f" | skipped {skipped}"
        line += f" | {(time.time() - t_start) / 60:.1f} min"

        print(line, flush=True)
        history.append(log)

        if stop:
            print(f"Early stop: no improvement for {PATIENCE} epochs.")
            break

    model.load_state_dict(best_state, strict=False)          # back to the best epoch

    return history, best_epoch

### CELL 12 · Train (train data only, best epoch picked on dev)

**In simple words:** Run the training: train files only, the dev files pick the best epoch.

In [12]:
# ============================================================
# CELL 12 — TRAIN (TRAIN data only, best epoch picked on dev)
# ============================================================

print(f"{RUN_NAME} | train {len(train_docs)} docs -> dev {len(dev_docs)} docs")

if DEVICE.type == "cuda":
    torch.cuda.reset_peak_memory_stats()

t0 = time.time()
history, best_epoch = train_model(model, EPOCHS)
train_minutes = (time.time() - t0) / 60

peak_gpu_gb = torch.cuda.max_memory_allocated() / 1e9 if DEVICE.type == "cuda" else 0.0

torch.save(trainable_state(model), OUTPUT_DIR / "best_trainable.pt")

print(f"\n✅ Trained in {train_minutes:.1f} min | best epoch {best_epoch} | peak GPU {peak_gpu_gb:.2f} GB")

p3_arch2_ia3 | train 80 docs -> dev 10 docs
Epoch  1/15 (head only) | loss 3.0856 | train F1 0.2223 | dev F1 CL 0.1804 IT 0.1695 mean 0.1750  <- best | 0.9 min
Epoch  2/15 | loss 2.5338 | train F1 0.1927 | dev F1 CL 0.2424 IT 0.1982 mean 0.2203  <- best | 1.8 min
Epoch  3/15 | loss 2.0880 | train F1 0.2514 | dev F1 CL 0.2654 IT 0.2175 mean 0.2414  <- best | 2.8 min
Epoch  4/15 | loss 1.8520 | train F1 0.2761 | dev F1 CL 0.2950 IT 0.2583 mean 0.2766  <- best | 3.7 min
Epoch  5/15 | loss 1.6237 | train F1 0.3124 | dev F1 CL 0.2661 IT 0.3304 mean 0.2983  <- best | 4.6 min
Epoch  6/15 | loss 1.4574 | train F1 0.3699 | dev F1 CL 0.2887 IT 0.2909 mean 0.2898 | 5.5 min
Epoch  7/15 | loss 1.2861 | train F1 0.4051 | dev F1 CL 0.3165 IT 0.3226 mean 0.3196  <- best | 6.5 min
Epoch  8/15 | loss 1.1416 | train F1 0.4751 | dev F1 CL 0.2695 IT 0.2948 mean 0.2822 | 7.4 min
Epoch  9/15 | loss 1.0128 | train F1 0.4451 | dev F1 CL 0.2850 IT 0.3475 mean 0.3162 | 8.3 min
Epoch 10/15 | loss 0.8588 | train F

### CELL 13 · Dev report: per domain, per label, confusions + save results

**In simple words:** Dev scores per domain and per role, and which roles get confused — saved for the report.

In [13]:
# ============================================================
# CELL 13 — DEV REPORT: per domain, per label, confusions + save results
# ============================================================

t0 = time.time()
dev_scores, dev_predictions = evaluate(model, dev_docs)
predict_seconds = time.time() - t0

print(f"DEV ({RUN_NAME}, best epoch {best_epoch})")
print(f"{'':6s} {'macro-P':>8s} {'macro-R':>8s} {'macro-F1':>9s}")

for key in ["CL", "IT", "all"]:
    if key in dev_scores:
        s = dev_scores[key]
        print(f"{key:6s} {s['precision']:8.4f} {s['recall']:8.4f} {s['f1']:9.4f}")


# ------------------------------------------------------------
# Per label + most frequent mistakes
# ------------------------------------------------------------

per_label = {}

for domain in ["CL", "IT"]:

    gold, pred = flat_labels(dev_docs, dev_predictions, domain)

    if not gold:
        continue

    p, r, f, support = precision_recall_fscore_support(gold, pred, labels=LABELS, zero_division=0)

    per_label[domain] = {
        label: {"precision": float(p[k]), "recall": float(r[k]), "f1": float(f[k]), "support": int(support[k])}
        for k, label in enumerate(LABELS)
    }

    print(f"\n{domain}: {'label':24s} {'P':>6s} {'R':>6s} {'F1':>6s} {'gold':>5s}")

    for label in LABELS:
        s = per_label[domain][label]
        print(f"    {label:24s} {s['precision']:6.3f} {s['recall']:6.3f} {s['f1']:6.3f} {s['support']:5d}")

    confusions = Counter((g, q) for g, q in zip(gold, pred) if g != q)

    print(f"\n  most frequent mistakes in {domain} (gold -> predicted):")

    for (g, q), count in confusions.most_common(8):
        print(f"    {g:24s} -> {q:24s} {count:4d}")


# ------------------------------------------------------------
# Closely related roles
# ------------------------------------------------------------

RELATED_ROLES = {
    "precedents": ["PrecedentReliedUpon", "PrecedentNotReliedUpon", "PrecedentOverruled"],
    "arguments": ["ArgumentPetitioner", "ArgumentRespondent"],
    "decision": ["RatioOfTheDecision", "RulingByPresentCourt"],
    "background": ["Fact", "RulingByLowerCourt"],
}

print(f"\n{'group':12s} {'domain':6s} {'sentences':>9s} {'correct':>8s} {'confused within group':>22s}")

confusion_groups = {}

for group, roles in RELATED_ROLES.items():
    for domain in ["CL", "IT"]:

        gold, pred = flat_labels(dev_docs, dev_predictions, domain)
        pairs = [(g, q) for g, q in zip(gold, pred) if g in roles]

        if not pairs:
            continue

        correct = sum(g == q for g, q in pairs) / len(pairs)
        within = sum(g != q and q in roles for g, q in pairs) / len(pairs)

        confusion_groups[f"{group}_{domain}"] = {"sentences": len(pairs), "correct": correct, "within_group": within}

        print(f"{group:12s} {domain:6s} {len(pairs):9d} {correct:8.1%} {within:22.1%}")


# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

results = {
    "run": RUN_NAME,
    "model": "Architecture-2 encoder (frozen) + IA3",
    "method": "ia3",
    "trained_on": "train only (dev used to pick the best epoch)",
    "parameters_total": n_total,
    "parameters_trainable": n_trainable,
    "parameters_adapter": n_adapter,
    "parameters_head": n_head,
    "best_epoch": best_epoch,
    "train_minutes": train_minutes,
    "peak_gpu_gb": peak_gpu_gb,
    "dev_predict_seconds": predict_seconds,
    "max_sentence_tokens": MAX_SENT_TOKENS,
    "window": WINDOW,
    "dev": dev_scores,
    "dev_per_label": per_label,
    "dev_related_roles": confusion_groups,
    "never_predicted": {d: [l for l, ok in zip(LABELS, LABEL_ALLOWED[d]) if not ok] for d in ["CL", "IT"]},
    "history": history,
}

with open(OUTPUT_DIR / "results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)

with open(OUTPUT_DIR / "dev_prediction.json", "w", encoding="utf-8") as f:
    json.dump(dev_predictions, f, ensure_ascii=False, indent=1)

print("\nSaved:", OUTPUT_DIR / "results.json", "and dev_prediction.json")

DEV (p3_arch2_ia3, best epoch 13)
        macro-P  macro-R  macro-F1
CL       0.4117   0.3699    0.3203
IT       0.5801   0.3577    0.3917
all      0.5857   0.3530    0.3736

CL: label                         P      R     F1  gold
    Fact                      0.408  0.570  0.476   249
    Issue                     0.000  0.000  0.000     3
    ArgumentPetitioner        0.610  0.137  0.223   344
    ArgumentRespondent        1.000  0.106  0.192   236
    Statute                   0.615  0.828  0.706    58
    Dissent                   0.000  0.000  0.000     0
    PrecedentReliedUpon       0.368  0.246  0.295   142
    PrecedentNotReliedUpon    0.000  0.000  0.000     0
    PrecedentOverruled        0.000  0.000  0.000     0
    RulingByLowerCourt        0.000  0.000  0.000    56
    RatioOfTheDecision        0.248  0.646  0.358   288
    RulingByPresentCourt      0.667  0.667  0.667    18
    None                      0.200  0.500  0.286     2

  most frequent mistakes in CL (gold -> 

### CELL 14 · Predict test + write prediction.json + submission.zip

**In simple words:** Predict the roles of every test document and write `submission.zip`.

**How it works:** `prediction.json` maps every document id to its ordered list of roles, one per sentence.

In [14]:
# ============================================================
# CELL 14 — PREDICT TEST + WRITE prediction.json + submission.zip
# ============================================================

test_predictions = predict_labels(model, test_docs)


# ------------------------------------------------------------
# Format checks: one valid label per sentence, same order, every document
# ------------------------------------------------------------

ids = [doc["id"] for doc in test_docs]
assert len(ids) == len(set(ids)), "the same document id appears in CL and IT test files"

for doc in test_docs:

    labels = test_predictions[doc["id"]]

    assert len(labels) == len(doc["sentences"]), doc["id"]
    assert all(label in LABEL2ID for label in labels), doc["id"]

print(f"✅ {len(test_docs)} test documents, {sum(len(v) for v in test_predictions.values())} sentences, all checks passed")

print("\nPredicted label distribution (test):")
for label, count in Counter(l for v in test_predictions.values() for l in v).most_common():
    print(f"  {label:24s} {count:6d}")


PREDICTION_FILE = OUTPUT_DIR / "prediction.json"
ZIP_FILE = OUTPUT_DIR / "submission.zip"

with open(PREDICTION_FILE, "w", encoding="utf-8") as f:
    json.dump(test_predictions, f, ensure_ascii=False, indent=1)

# the same predictions split by domain (in case Codabench wants one file per domain)
for domain in ["CL", "IT"]:
    with open(OUTPUT_DIR / f"{domain}_prediction.json", "w", encoding="utf-8") as f:
        json.dump({d["id"]: test_predictions[d["id"]] for d in test_docs if d["domain"] == domain},
                  f, ensure_ascii=False, indent=1)

with zipfile.ZipFile(ZIP_FILE, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(PREDICTION_FILE, arcname="prediction.json")        # file at the ROOT of the zip

with zipfile.ZipFile(ZIP_FILE) as zf:
    print("\nsubmission.zip:", zf.namelist())

print("Saved:", ZIP_FILE)

✅ 10 test documents, 2197 sentences, all checks passed

Predicted label distribution (test):
  RatioOfTheDecision          882
  Fact                        507
  PrecedentReliedUpon         281
  Statute                     277
  ArgumentPetitioner          174
  ArgumentRespondent           34
  RulingByPresentCourt         27
  PrecedentNotReliedUpon        7
  RulingByLowerCourt            5
  None                          3

submission.zip: ['prediction.json']
Saved: /kaggle/working/p3_arch2_ia3/submission.zip
